# Feature Selection Pipeline

Three sequential steps:
1. **Step 1** – Remove invalid / low-variance features
2. **Step 2** – Pearson correlation redundancy removal
3. **Step 3** – Boruta importance-based selection (per target)

All logic lives in `feature_selection.py`; this notebook only orchestrates calls.

In [ ]:
import numpy as np

import preprocess_ML as pp
import feature_selection as fs
from visualization import plot_retained_features
from config import (
    DEFAULT_TASK, DEFAULT_DATASET,
    MORGAN_RADIUS,
    VARIANCE_THRESHOLD, ZERO_RATIO_THRESHOLD, PEARSON_THRESHOLD,
    BORUTA_N_ESTIMATORS, BORUTA_MAX_ITER, BORUTA_RANDOM_STATE,
)

## 0. Load raw dataset

In [ ]:
(X_train, y_train_est, y_train_krisc,
 X_dev,   y_dev_est,   y_dev_krisc,
 X_test,  y_test_est,  y_test_krisc,
 feature_names) = pp.create_datasets(
    task=DEFAULT_TASK,
    dataset=DEFAULT_DATASET,
    radius=MORGAN_RADIUS,
    device=None,
)

# Persist label arrays once (shared by both notebooks)
fs.save_labels(
    y_train_est, y_train_krisc,
    y_dev_est,   y_dev_krisc,
    y_test_est,  y_test_krisc,
)

## 1. Remove invalid features

In [ ]:
(X_train_s1, X_dev_s1, X_test_s1,
 names_s1, report_s1) = fs.step1_remove_invalid_features(
    X_train, X_dev, X_test, feature_names,
    variance_threshold=VARIANCE_THRESHOLD,
    zero_ratio_threshold=ZERO_RATIO_THRESHOLD,
)
fs.save_step1(X_train_s1, X_dev_s1, X_test_s1, names_s1)

## 2. Pearson redundancy removal

In [ ]:
(X_train_s2, X_dev_s2, X_test_s2,
 names_s2, report_s2) = fs.step2_pearson_redundancy(
    X_train_s1, X_dev_s1, X_test_s1, names_s1,
    corr_threshold=PEARSON_THRESHOLD,
)
fs.save_step2(X_train_s2, X_dev_s2, X_test_s2, names_s2)

print(f"\n原始特征: 2432 → Step 1 后: {len(names_s1)} → Step 2 后: {len(names_s2)}")

## 3. Boruta feature selection

In [ ]:
names_s2_arr = np.array(names_s2)

est_confirmed, est_tentative, est_rejected, est_ranking = fs.run_boruta(
    X_train_s2, y_train_est, names_s2_arr,
    target_name='ΔEST',
    n_estimators=BORUTA_N_ESTIMATORS,
    max_iter=BORUTA_MAX_ITER,
    random_state=BORUTA_RANDOM_STATE,
)

In [ ]:
krisc_confirmed, krisc_tentative, krisc_rejected, krisc_ranking = fs.run_boruta(
    X_train_s2, y_train_krisc, names_s2_arr,
    target_name='kRISC',
    n_estimators=BORUTA_N_ESTIMATORS,
    max_iter=BORUTA_MAX_ITER,
    random_state=BORUTA_RANDOM_STATE,
)

## 4. Apply Boruta filter & save

In [ ]:
(X_tr_est, X_dv_est, X_te_est,
 names_est) = fs.apply_boruta_filter(
    X_train_s2, X_dev_s2, X_test_s2, names_s2_arr,
    est_confirmed, est_tentative, target_name='ΔEST',
)
fs.save_boruta('est', X_tr_est, X_dv_est, X_te_est, names_est)

(X_tr_krisc, X_dv_krisc, X_te_krisc,
 names_krisc) = fs.apply_boruta_filter(
    X_train_s2, X_dev_s2, X_test_s2, names_s2_arr,
    krisc_confirmed, krisc_tentative, target_name='kRISC',
)
fs.save_boruta('krisc', X_tr_krisc, X_dv_krisc, X_te_krisc, names_krisc)

print(f"\nΔEST Boruta后: {len(names_est)} 个特征")
print(f"kRISC Boruta后: {len(names_krisc)} 个特征")

## 5. Visualise retained features

In [ ]:
plot_retained_features(
    X_train_s2, y_train_est, names_s2_arr,
    confirmed_list=est_confirmed,
    tentative_list=est_tentative,
    target_name='ΔEST',
    filename='retained_features_score_est.png',
)

In [ ]:
plot_retained_features(
    X_train_s2, y_train_krisc, names_s2_arr,
    confirmed_list=krisc_confirmed,
    tentative_list=krisc_tentative,
    target_name='kRISC',
    filename='retained_features_score_krisc.png',
)